# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cokezero20/FlyRank_AI_ML_Internship_NATIVIDAD/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

### What this queue is

A prioritized list of content items to review, ranked by the ones most likely to be declining. Each item carries a reason code explaining why it was flagged, so the reviewer knows what to look at — not just that something appeared on the list.

### Why the baseline rule, not the model

The Random Forest model scored well on random splits (AUC 0.754) but dropped to 0.549 under grouped validation — barely above a coin flip on unseen clients. A playbook built on unreliable scores would send reviewers to the wrong pages. The baseline rule (score by search position, filtered to visible content) is transparent, doesn't memorize clients, and beat the base rate honestly. Its known weakness — confusing "never performed" with "declining" — is something a human reviewer can catch in seconds. The model's weakness — client memorization — is invisible to a reviewer.

### What to do first, and why

| Priority | What it means | Why these first | How many |
|---|---|---|---|
| P1 | Review within 1 week | Highest-scoring items — worst position among visible content. These are the pages most likely losing ground in search right now. | Top 50 |
| P2 | Review within 2 weeks | Still flagged, lower urgency. Worth reviewing but less likely to need immediate action. | Next 150 |
| P3 | Watch list | Visible content with a score, but not urgent. Add to a monitoring cycle rather than acting immediately. | Remaining scored items |
| Skip | No action | Below the visibility threshold (fewer than 100 impressions). No measurable signal to act on — reviewing these wastes time. | Unscored items |

### Reason codes — what the reviewer should look for

| Reason code | What it means in plain words | Reviewer action |
|---|---|---|
| low_ctr_for_position | This page gets seen in search results but almost nobody clicks it. Its position is poor and it's not earning attention. | Check the title, meta description, and whether the page still matches the search intent it ranks for. |
| below_visibility_threshold | This page has fewer than 100 impressions. There isn't enough data to judge whether it's declining or just invisible. | Skip unless you have a strategic reason to investigate. |

### Decay/refresh insight

In the FlyRank research paper, content was observed to peak at 61–90 days and decline after 270 days. Old content refreshed within 30 days showed materially higher impressions than unrefreshed old content. This comparison lacks a matched control group, so the size of the effect is uncertain — but the direction is consistent. P1 items older than 270 days should be considered for a content refresh, not just a title/snippet review.

### Archetype guide — what kind of page is this?

| If the page looks like this | It probably means | Do this |
|---|---|---|
| High position number, low CTR, 100+ impressions | Visible but not earning clicks — the core decline signal | Review title, description, and intent match |
| Old (270+ days), no recent update | Entering the observed decay zone | Prioritize for content refresh — update facts, add missing sections |
| Low impressions, bad position | Never performed. Not declining, just weak | Skip or monitor. Do not treat as declining. |
| Good position, good CTR, still flagged as declining | Decline from causes outside the feature set | Flag for manual investigation — may be seasonal, competitive, or algorithmic |

In [1]:
import pandas as pd
import numpy as np
from datetime import date
from datasets import load_dataset
from google.colab import userdata
import os

SEED = 42
HF_TOKEN = userdata.get('HF_Token')

# ── Load data ────────────────────────────────────────────────────────
dataset = load_dataset(
    'FlyRank/internship-warehouse',
    name='fact_content_daily_performance',
    token=HF_TOKEN,
    streaming=True
)

print("Loading January–April 2026 data...")
data_rows = []
for batch in dataset['train'].iter(batch_size=100000):
    batch_df = pd.DataFrame(batch)
    if isinstance(batch_df['report_date'].iloc[0], str):
        batch_df['report_date'] = pd.to_datetime(batch_df['report_date']).dt.date
    data_batch = batch_df[
        (batch_df['report_date'] >= date(2026, 1, 1)) &
        (batch_df['report_date'] <= date(2026, 4, 30)) &
        (batch_df['ga4_data_available'] == True)
    ]
    if len(data_batch) > 0:
        data_rows.append(data_batch)

df = pd.concat(data_rows, ignore_index=True)
df['month'] = pd.to_datetime(df['report_date']).dt.month
print(f"Total rows: {len(df):,}")

# ── Create label ─────────────────────────────────────────────────────
monthly_impr = (
    df[df['month'].isin([3, 4])]
    .groupby(['content_hash_id', 'month'])['gsc_impressions']
    .sum()
    .unstack(fill_value=0)
)
monthly_impr.columns = ['mar_impressions', 'apr_impressions']
monthly_impr['is_declining_label'] = (
    monthly_impr['apr_impressions'] < (0.8 * monthly_impr['mar_impressions'])
).astype(int)

# ── Build content-level features (Jan–Mar) ────────────────────────────
df_features = df[df['month'].isin([1, 2, 3])]

content = df_features.groupby('content_hash_id').agg(
    total_impressions=('gsc_impressions', 'sum'),
    total_clicks=('gsc_clicks', 'sum'),
    avg_position=('gsc_avg_position', 'mean'),
    total_sessions=('ga4_sessions', 'sum'),
    total_engaged=('ga4_engaged_sessions', 'sum'),
    total_pageviews=('ga4_pageviews', 'sum'),
    days_observed=('report_date', 'nunique')
).reset_index()

content['ctr'] = content['total_clicks'] / content['total_impressions'].replace(0, np.nan)
content['ctr'] = content['ctr'].fillna(0)

# Merge label
content = content.merge(monthly_impr[['is_declining_label']], on='content_hash_id', how='inner')

# ── Apply baseline rule ───────────────────────────────────────────────
content['visible'] = (content['total_impressions'] >= 100).astype(int)
content['score'] = content['avg_position'] * content['visible']

# Reason code
content['reason_code'] = np.where(
    content['visible'] == 1,
    'low_ctr_for_position',
    'below_visibility_threshold'
)

# Rank and assign priority
content = content.sort_values('score', ascending=False).reset_index(drop=True)
content['rank'] = content.index + 1

content['action'] = np.where(content['visible'] == 0, 'no_action',
                   np.where(content['rank'] <= 50, 'review_and_optimize',
                   np.where(content['rank'] <= 200, 'review_and_optimize',
                   'monitor')))

content['priority'] = np.where(content['visible'] == 0, 'Skip',
                     np.where(content['rank'] <= 50, 'P1',
                     np.where(content['rank'] <= 200, 'P2',
                     'P3')))

# ── Print what a human needs to see ───────────────────────────────────
print("\n" + "=" * 60)
print("ACTION QUEUE: What to do first")
print("=" * 60)

for p, label in [('P1', 'Review within 1 week'),
                 ('P2', 'Review within 2 weeks'),
                 ('P3', 'Watch list'),
                 ('Skip', 'No action')]:
    subset = content[content['priority'] == p]
    rate = subset['is_declining_label'].mean() if len(subset) > 0 else 0
    print(f"\n  {p} — {label}")
    print(f"    Items:        {len(subset):,}")
    print(f"    Decline rate: {rate:.1%}")
    if p != 'Skip':
        print(f"    Reason:       {subset['reason_code'].iloc[0] if len(subset) > 0 else 'n/a'}")

base_rate = content['is_declining_label'].mean()
print(f"\n  Base rate (all items): {base_rate:.1%}")
print(f"  P1 precision:         {content.head(50)['is_declining_label'].mean():.1%}")
print(f"  P2 precision:         {content.head(200)['is_declining_label'].mean():.1%}")

# Show the top 10 so the reviewer knows what P1 looks like
print("\n" + "=" * 60)
print("TOP 10 (P1) — what these pages look like")
print("=" * 60)
top10 = content.head(10)[['rank', 'content_hash_id', 'score', 'reason_code',
                           'action', 'avg_position', 'total_impressions',
                           'ctr', 'is_declining_label']]
print(top10.to_string(index=False))


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading January–April 2026 data...
Total rows: 1,199,364

ACTION QUEUE: What to do first

  P1 — Review within 1 week
    Items:        50
    Decline rate: 68.0%
    Reason:       low_ctr_for_position

  P2 — Review within 2 weeks
    Items:        150
    Decline rate: 71.3%
    Reason:       low_ctr_for_position

  P3 — Watch list
    Items:        34,824
    Decline rate: 49.7%
    Reason:       low_ctr_for_position

  Skip — No action
    Items:        60,425
    Decline rate: 29.8%

  Base rate (all items): 37.2%
  P1 precision:         68.0%
  P2 precision:         70.5%

TOP 10 (P1) — what these pages look like
 rank          content_hash_id     score          reason_code              action  avg_position  total_impressions      ctr  is_declining_label
    1 content_c26c91a74fe92a59 98.951069 low_ctr_for_position review_and_optimize     98.951069              140.0 0.007143                   1
    2 content_28604ef85fd52163 82.724432 low_ctr_for_position review_and_optimize    

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Who uses this

**Primary user:** A content strategist or SEO editor responsible for deciding which pages to refresh, rewrite, or consolidate. This person has access to the actual pages and can judge whether a flagged item needs work or is a false alarm.

**Secondary user:** A content team lead allocating reviewer time across a portfolio. The priority tiers (P1/P2/P3) help them plan weekly and biweekly review cycles without scanning the full inventory.

**Not intended for:** Engineers building automated content pipelines, executives making budget decisions based on decline projections, or anyone treating this queue as a final answer rather than a starting point for review.

### For what

This is a **decision-support tool for content review prioritization**. It ranks content items by how likely they are to be declining, so a reviewer can start with the most urgent pages instead of scanning the entire portfolio. The reviewer makes the final call on every action.

**Intended workflow:**
1. A content reviewer opens the ranked queue
2. They start with P1 items — 50 pages most likely declining
3. For each page, they read the reason code, check the page, and decide: refresh, rewrite, consolidate, or skip
4. P2 items are reviewed in the next cycle
5. P3 items are monitored passively — no immediate action unless something changes

### Where it stops being valid

**Different time period.** This queue scores content using January–March 2026 data to flag April 2026 decline. It has not been tested on other quarters. Seasonal patterns, algorithm updates, or market shifts in a different window could change which signals matter.

**Different portfolio.** The data covers 57 brands. A portfolio with different industries, content types, or audience sizes may produce different position-to-decline relationships. The rule's thresholds (100-impression visibility cutoff, position-based scoring) were tuned to this dataset.

**Unseen clients.** The Random Forest model dropped to AUC 0.549 on clients it had never trained on. The baseline rule is more stable but has not been formally tested on held-out clients either. Treat all scores as directional, not guaranteed.

**Causal claims.** This queue identifies content that is *associated with* decline patterns. It does not explain *why* content declines or prove that any specific action will reverse it. The honest form is: "these pages look worth reviewing first, because their position and visibility pattern matches what declining content looked like in this dataset."

**Automation.** This queue is not validated for automated action. No page should be depublished, redirected, or rewritten by a script based on this score alone.

### Known limits

| Limit | What it means for the reviewer |
|---|---|
| The rule confuses "never performed" with "declining" | ~32% of P1 items are not actually declining. The reviewer should check: did this page ever have meaningful traffic, or has it always been weak? |
| Single reason code | Every flagged item gets `low_ctr_for_position`. The rule cannot distinguish between a title problem, a content staleness problem, or a competition problem. The reviewer must diagnose the cause. |
| No trend information | The score uses average position, not position *change*. A page stuck at position 80 for six months scores the same as one that dropped from position 10 to 80 last month. |
| 57-brand portfolio only | Patterns observed here may not hold for different industries, content types, or portfolio sizes. |
| Snapshot, not longitudinal | This queue reflects one time window. Content that enters the decline zone after April 2026 is not captured. |

### Cost/value thinking

Reviewing 50 pages (P1) at roughly 10 minutes per page costs about 8 hours of reviewer time. At 68% precision, approximately 34 of those pages are genuinely declining and worth acting on. The value depends on what those pages are worth — but spending 8 hours to find 34 high-priority refresh candidates is a reasonable trade in most content operations.

Reviewing all 200 (P1 + P2) costs roughly 33 hours. At 70.5% precision, approximately 141 are genuinely declining. Whether that's worth it depends on team capacity and the value density of the portfolio.

Reviewing P3 (34,824 items) is not recommended without further filtering. At 49.7% decline rate — close to the base rate — the signal is too weak to justify the time.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### What a person must check before acting on any flagged item

Every item in this queue is a suggestion, not an instruction. Before taking action on any P1 or P2 item, the reviewer should answer these five questions:

**Question 1: Is this page actually declining, or was it always weak?**
The rule's known blind spot. Check the page's traffic history over the past 6 months. If impressions have been flat near zero the entire time, this is not a decline — it's a page that never gained traction. Skip it.

**Question 2: Does the page still match its target search intent?**
Open the page. Search the query it ranks for. Read the top 3 competing results. If the page no longer answers what searchers are looking for, a title fix won't help — the content itself needs updating or the page needs consolidating with a stronger one.

**Question 3: Is the decline seasonal or temporary?**
Some content declines every year at the same time (holiday content in January, tax content after April). If the page declined at the same time last year and recovered, this is not a problem to fix — it's a pattern to expect. Mark it as seasonal and skip.

**Question 4: Is there a strategic reason to keep this page as-is?**
Some pages exist for brand, legal, or partnership reasons regardless of search performance. If the page serves a purpose beyond organic traffic, the reviewer should note that and skip.

**Question 5: Am I the right person to act on this?**
If the fix requires engineering (redirect, canonical tag, schema markup), design (page layout, UX), or legal review (claims, compliance), escalate instead of editing. The queue tells you where to look, not who should fix it.

### The review checklist (one pass per item)

| Step | What to do | Time |
|---|---|---|
| 1 | Check 6-month traffic trend — declining or always flat? | 1 min |
| 2 | Search the target query — does the page match intent? | 2 min |
| 3 | Read the page — is the content stale, thin, or outdated? | 3 min |
| 4 | Check for seasonality or strategic hold | 1 min |
| 5 | Decide: refresh, rewrite, consolidate, escalate, or skip | 1 min |
| 6 | Log the decision and reason | 1 min |

Estimated time per item: ~10 minutes. P1 (50 items) takes roughly one working day.

### What should NEVER be automated

| Action | Why it must stay manual |
|---|---|
| **Depublishing or deleting a page** | A false positive means deleting a page that was not declining. The rule's 32% false positive rate at P1 makes automated deletion unacceptable. A deleted page with existing backlinks or indexed URLs causes downstream damage that is expensive to reverse. |
| **Redirecting a URL** | Redirects are permanent architectural decisions. A bad redirect breaks user bookmarks, confuses crawlers, and loses link equity. This requires a human understanding the relationship between the source and destination pages. |
| **Rewriting content** | The queue does not know what's wrong with the content — only that its position and CTR pattern matches decline. Automated rewriting without understanding the cause would produce generic output that may make the page worse. |
| **Consolidating or merging pages** | Merging two pages requires editorial judgment about which content to keep, which URL to preserve, and how to handle overlapping keywords. The queue cannot make these decisions. |
| **Changing page titles or meta descriptions at scale** | Bulk title changes based on a score risk damaging pages that are performing fine. Titles should be reviewed individually against the actual search queries the page ranks for. |
| **Acting on P3 items without further filtering** | P3's decline rate (49.7%) is close to the base rate (37.2%). Automating actions on 34,824 items with near-random signal would waste resources and damage healthy pages. |

### The one-sentence rule

**If the action is irreversible or expensive to undo, a human must review it first.** The queue's job is to save the reviewer time finding what to look at — not to replace the reviewer's judgment about what to do.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

This queue was built on one snapshot: January–April 2026, 57 brands. It will go stale. The question is not whether, but when — and whether you notice before acting on outdated priorities. Here are the signals that tell you the queue no longer reflects reality.

### Staleness signals — check monthly

| Signal | What to measure | Stale threshold | Why it matters |
|---|---|---|---|
| **P1 precision decay** | Of the P1 items reviewed this month, what fraction were genuinely declining? | Drops below 50% | At 50%, the queue is barely better than flipping a coin against the base rate. Reviewer time is being wasted on false alarms. |
| **Base rate shift** | What is the current portfolio-wide decline rate? | Moves more than 10 points from the original 37.2% | If the overall decline rate changes substantially (e.g., a major algorithm update causes widespread drops, or the portfolio grows healthier), the queue's thresholds no longer fit the landscape. |
| **Position distribution shift** | What does the distribution of avg_position look like now vs January–March 2026? | Median position shifts by more than 5 points | The rule scores on position. If the portfolio's position profile changes materially, the same score means something different than it did when the queue was built. |
| **New content volume** | How much of the current portfolio was published after April 2026? | More than 30% of active content is post-April | New content has no representation in this queue. If a third of the portfolio is new, the queue is missing a large segment. |
| **Client mix change** | Have new clients been added or existing ones removed? | More than 20% client turnover | The model showed strong client-dependent behavior. New clients bring patterns the rule has never been evaluated against. |

### When to rebuild, not just re-run

Re-running the same rule on new data is not the same as validating it still works. Rebuild when:

**Trigger 1: Precision drops below 50% for two consecutive months.**
The rule's signal has degraded. Recheck whether position is still the strongest indicator of decline, or whether other features (impressions trend, engagement change) have become more predictive.

**Trigger 2: A major Google algorithm update occurs.**
Algorithm updates can change which content declines and why. The position-to-decline relationship that held in Q1 2026 may not hold after an update. Re-run the signal checks (bucket tables from Week 4) on fresh data before trusting the existing queue.

**Trigger 3: The portfolio changes shape.**
If the brand count doubles, the content mix shifts (e.g., from mostly informational to mostly commercial), or a large batch of new content is published, the patterns in this queue may no longer represent the portfolio. Rebuild the label, recheck signals, and re-score.

**Trigger 4: The decline definition changes.**
The current label is "April impressions < 80% of March impressions." If the business decides that a different threshold (70%, 90%), a different metric (clicks instead of impressions), or a different window (60-day instead of 30-day) better defines "declining," the entire queue must be rebuilt from the label up.

### What monitoring looks like in practice

| Frequency | Action | Who |
|---|---|---|
| Weekly | Log reviewer decisions on P1 items: acted on, skipped, false alarm | Content reviewer |
| Monthly | Calculate precision from logged decisions. Compare against 68% baseline. | Team lead |
| Monthly | Check portfolio base rate and position distribution for drift | Analyst |
| Quarterly | Full rebuild: re-run signal checks, re-score queue, revalidate precision@K on fresh data | Analyst |

### What this monitoring is NOT

This is a manual check cycle, not an automated alerting system. There is no production pipeline watching these metrics. The analyst runs the checks, compares the numbers, and decides whether to rebuild. This is appropriate for a non-production decision-support tool. If this queue were ever promoted to a production system, automated drift detection would be required — but that is outside the scope of this playbook.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

These files are the deliverables that the research paper will build on. Everything lands in `work/outputs/`. The queue is the primary export. The summary tables and figures support the recommendations section of the paper.

In [5]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

os.makedirs('work/outputs', exist_ok=True)

# ── Export 1: The ranked action queue ──────────────────────────────────
queue_cols = ['rank', 'content_hash_id', 'score', 'reason_code', 'action',
              'priority', 'avg_position', 'total_impressions', 'total_clicks',
              'ctr', 'total_sessions', 'days_observed', 'is_declining_label']

content[queue_cols].to_csv('work/outputs/playbook_action_queue.csv', index=False)
print(f"✓ Exported: playbook_action_queue.csv ({len(content):,} rows)")

# ── Export 2: Priority summary table ──────────────────────────────────
summary = content.groupby('priority').agg(
    count=('content_hash_id', 'count'),
    decline_rate=('is_declining_label', 'mean'),
    avg_score=('score', 'mean'),
    avg_position=('avg_position', 'mean'),
    avg_impressions=('total_impressions', 'mean'),
    avg_ctr=('ctr', 'mean')
).round(3)

summary.to_csv('work/outputs/playbook_priority_summary.csv')
print(f"✓ Exported: playbook_priority_summary.csv")
print(summary.to_string())

# ── Export 3: Figure — Precision by priority tier ─────────────────────
tiers = ['P1', 'P2', 'P3', 'Skip']
precisions = []
counts = []
for t in tiers:
    subset = content[content['priority'] == t]
    precisions.append(subset['is_declining_label'].mean())
    counts.append(len(subset))

base_rate = content['is_declining_label'].mean()

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(tiers, precisions, color=['#c0392b', '#e67e22', '#f1c40f', '#bdc3c7'])
ax.axhline(y=base_rate, color='black', linestyle='--', linewidth=1, label=f'Base rate ({base_rate:.1%})')
ax.set_ylabel('Decline Rate')
ax.set_title('Decline Rate by Priority Tier vs Base Rate')
ax.set_ylim(0, 1)
ax.legend()

for bar, p, n in zip(bars, precisions, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{p:.1%}\n(n={n:,})', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('work/outputs/fig_precision_by_tier.png', dpi=150)
plt.close()
print("✓ Exported: fig_precision_by_tier.png")

# ── Export 4: Figure — Top 50 score distribution ──────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
top50 = content.head(50)
colors = ['#c0392b' if d == 1 else '#2ecc71' for d in top50['is_declining_label']]
ax.barh(range(50, 0, -1), top50['score'].values, color=colors)
ax.set_ylabel('Rank')
ax.set_xlabel('Score (avg_position × visible)')
ax.set_title('P1 Items: Score Distribution (red = declining, green = stable)')
ax.set_yticks(range(50, 0, -10))
ax.set_yticklabels(range(1, 51, 10))
plt.tight_layout()
plt.savefig('work/outputs/fig_p1_score_distribution.png', dpi=150)
plt.close()
print("✓ Exported: fig_p1_score_distribution.png")

# ── Export 5: Figure — Precision@K curve ──────────────────────────────
ks = [10, 20, 50, 100, 200, 500, 1000]
prec_at_k = [content.head(k)['is_declining_label'].mean() for k in ks]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ks, prec_at_k, marker='o', color='#2c3e50', linewidth=2)
ax.axhline(y=base_rate, color='black', linestyle='--', linewidth=1, label=f'Base rate ({base_rate:.1%})')
ax.set_xlabel('K (number of items reviewed)')
ax.set_ylabel('Precision@K')
ax.set_title('Precision@K: How accurate is the queue at different review depths?')
ax.legend()
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig('work/outputs/fig_precision_at_k.png', dpi=150)
plt.close()
print("✓ Exported: fig_precision_at_k.png")

# ── Summary of all exports ────────────────────────────────────────────
print("\n" + "=" * 60)
print("ALL EXPORTS IN work/outputs/")
print("=" * 60)
for f in sorted(os.listdir('work/outputs')):
    size = os.path.getsize(f'work/outputs/{f}')
    print(f"  {f:<45} {size:>10,} bytes")


✓ Exported: playbook_action_queue.csv (95,449 rows)
✓ Exported: playbook_priority_summary.csv
          count  decline_rate  avg_score  avg_position  avg_impressions  avg_ctr
priority                                                                        
P1           50         0.680     67.918        67.918          484.820    0.004
P2          150         0.713     51.691        51.691         2050.647    0.003
P3        34824         0.497     11.484        11.484         3682.313    0.008
Skip      60425         0.298      0.000        14.295           13.806    0.031
✓ Exported: fig_precision_by_tier.png
✓ Exported: fig_p1_score_distribution.png
✓ Exported: fig_precision_at_k.png

ALL EXPORTS IN work/outputs/
  fig_p1_score_distribution.png                     32,456 bytes
  fig_precision_at_k.png                            48,322 bytes
  fig_precision_by_tier.png                         47,114 bytes
  playbook_action_queue.csv                     11,236,578 bytes
  playbook_prio

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.